# ⚙️ IPL Win Predictor: Feature Engineering & Model Training

This notebook demonstrates the feature engineering pipeline, the strict temporal train/test split to prevent data leakage, and the training of multiple classification models with probability calibration.

### Key Highlights:
1. **Zero Future Leakage:** Temporal split by season ($\le 2022$ train, $\ge 2023$ test).
2. **Feature Engineering:** Rate pressure delta (`rrr - crr`), wicket collapse rate, and innings phase binning.
3. **Model Exploration:**
   - Logistic Regression Baseline
   - Raw LightGBM Classifier
   - Calibrated LightGBM (Isotonic Regression)


In [1]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Scikit-Learn & LightGBM
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score, accuracy_score
import lightgbm as lgb

%matplotlib inline


## 1. Load Data & Add Derived Features

In [2]:
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds rate pressure delta, wicket collapse rate, and innings phase features.
    """
    df = df.copy()
    df["rrr_minus_crr"] = df["rrr"] - df["crr"]
    df["wicket_pressure"] = df["wickets_fallen"] / (df["legal_balls_bowled"] + 1)

    def get_phase(balls: int) -> str:
        if balls <= 36: return "Powerplay"
        elif balls <= 66: return "Middle_1"
        elif balls <= 90: return "Middle_2"
        elif balls <= 108: return "Death_1"
        else: return "Death_2"

    df["innings_phase"] = df["legal_balls_bowled"].apply(get_phase)
    return df


def split_train_test_by_season(df: pd.DataFrame, test_start_year: int = 2023):
    """
    Splits DataFrame strictly by season/year without leaking future data.
    Seasons <= 2022 -> Train | Seasons >= 2023 -> Test
    """
    def extract_year(s):
        s_str = str(s).strip()
        if "/" in s_str: return int(s_str.split("/")[0])
        try: return int(s_str)
        except ValueError: return 2008

    seasons_year = df["season"].apply(extract_year)
    df_train = df[seasons_year < test_start_year].copy()
    df_test = df[seasons_year >= test_start_year].copy()

    train_matches = set(df_train["match_id"].unique())
    test_matches = set(df_test["match_id"].unique())
    assert len(train_matches.intersection(test_matches)) == 0, "Match ID overlap detected!"

    print(f"Train matches: {len(train_matches)} | Train rows: {len(df_train)}")
    print(f"Test matches: {len(test_matches)} | Test rows: {len(df_test)}")
    return df_train, df_test

df = pd.read_csv('../data/processed/chases_ballbyball.csv', low_memory=False)
df = add_derived_features(df)

print(f"Total rows: {len(df):,}")
print("New derived features: 'rrr_minus_crr', 'wicket_pressure', 'innings_phase'")
df[['legal_balls_bowled', 'crr', 'rrr', 'rrr_minus_crr', 'wicket_pressure', 'innings_phase']].head()


Total rows: 137,887
New derived features: 'rrr_minus_crr', 'wicket_pressure', 'innings_phase'


,legal_balls_bowled,crr,rrr,rrr_minus_crr,wicket_pressure,innings_phase
0,0,0.0,10.4000,10.4000,0.0,Powerplay
1,1,6.0,10.4370,4.4370,0.0,Powerplay
2,2,3.0,10.5254,7.5254,0.0,Powerplay
3,3,2.0,10.6154,8.6154,0.0,Powerplay
4,4,4.5,10.6034,6.1034,0.0,Powerplay


## 2. Strict Temporal Train/Test Split (Preventing Data Leakage)
- **Train Set:** Seasons 2008 through 2022.
- **Test Set:** Seasons 2023 through 2026.
- Split is strictly by `match_id` — deliveries from the same match are never split across train and test.


In [3]:
df_train, df_test = split_train_test_by_season(df, test_start_year=2023)

numeric_features = [
    "runs_left", "balls_left", "wickets_in_hand",
    "rrr", "crr", "rrr_minus_crr", "target", "wicket_pressure"
]
categorical_features = ["batting_team", "bowling_team", "venue"]
feature_cols = numeric_features + categorical_features

X_train = df_train[feature_cols]
y_train = df_train["batting_team_won"]

X_test = df_test[feature_cols]
y_test = df_test["batting_team_won"]

print(f"Feature count: {len(feature_cols)} ({len(numeric_features)} numeric, {len(categorical_features)} categorical)")


Train matches: 906 | Train rows: 105437
Test matches: 279 | Test rows: 32450
Feature count: 11 (8 numeric, 3 categorical)


## 3. Model 1: Logistic Regression Baseline
The baseline uses standard scaling on numeric variables and one-hot encoding on categorical variables.
Logistic Regression outputs smooth, monotonic probabilities through the sigmoid function.


In [4]:
preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(max_iter=1000, solver="lbfgs", random_state=42))
])

lr_pipeline.fit(X_train, y_train)
lr_probs = lr_pipeline.predict_proba(X_test)[:, 1]

print("--- Logistic Regression Baseline Results ---")
print(f"Brier Score: {brier_score_loss(y_test, lr_probs):.4f}")
print(f"Log Loss:    {log_loss(y_test, lr_probs):.4f}")
print(f"ROC-AUC:     {roc_auc_score(y_test, lr_probs):.4f}")
print(f"Accuracy:    {accuracy_score(y_test, (lr_probs >= 0.5).astype(int)):.4f}")


--- Logistic Regression Baseline Results ---
Brier Score: 0.1887
Log Loss:    0.6167
ROC-AUC:     0.8357
Accuracy:    0.7442


## 4. Model 2: Raw LightGBM Classifier
Gradient Boosted Decision Trees excel at capturing non-linear interactions (e.g. death-over boundary hitting, batting collapses).


In [5]:
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

lgb_base = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

lgb_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("classifier", lgb_base)
])

lgb_pipeline.fit(X_train, y_train)
lgb_probs = lgb_pipeline.predict_proba(X_test)[:, 1]

print("--- Raw LightGBM Results ---")
print(f"Brier Score: {brier_score_loss(y_test, lgb_probs):.4f}")
print(f"Log Loss:    {log_loss(y_test, lgb_probs):.4f}")
print(f"ROC-AUC:     {roc_auc_score(y_test, lgb_probs):.4f}")
print(f"Accuracy:    {accuracy_score(y_test, (lgb_probs >= 0.5).astype(int)):.4f}")


--- Raw LightGBM Results ---
Brier Score: 0.2139
Log Loss:    0.8140
ROC-AUC:     0.8224
Accuracy:    0.7314


## 5. Model 3: Calibrated LightGBM (Isotonic Regression)
Tree-based models are notoriously prone to overconfidence near the 0 and 1 boundaries.
Wrapping the pipeline with `CalibratedClassifierCV(method='isotonic')` aligns predicted probabilities with true empirical win frequencies.


In [6]:
calibrated_lgb = CalibratedClassifierCV(
    estimator=lgb_pipeline,
    method="isotonic",
    cv=3
)
calibrated_lgb.fit(X_train, y_train)
cal_probs = calibrated_lgb.predict_proba(X_test)[:, 1]

print("--- Calibrated LightGBM (Isotonic) Results ---")
print(f"Brier Score: {brier_score_loss(y_test, cal_probs):.4f}")
print(f"Log Loss:    {log_loss(y_test, cal_probs):.4f}")
print(f"ROC-AUC:     {roc_auc_score(y_test, cal_probs):.4f}")
print(f"Accuracy:    {accuracy_score(y_test, (cal_probs >= 0.5).astype(int)):.4f}")


--- Calibrated LightGBM (Isotonic) Results ---
Brier Score: 0.1781
Log Loss:    0.7800
ROC-AUC:     0.8403
Accuracy:    0.7528


## 6. Model Comparison Summary Table

In [7]:
comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Raw LightGBM", "Calibrated LightGBM"],
    "Brier Score (lower is better)": [
        brier_score_loss(y_test, lr_probs),
        brier_score_loss(y_test, lgb_probs),
        brier_score_loss(y_test, cal_probs)
    ],
    "Log Loss (lower is better)": [
        log_loss(y_test, lr_probs),
        log_loss(y_test, lgb_probs),
        log_loss(y_test, cal_probs)
    ],
    "ROC-AUC (higher is better)": [
        roc_auc_score(y_test, lr_probs),
        roc_auc_score(y_test, lgb_probs),
        roc_auc_score(y_test, cal_probs)
    ],
    "Accuracy @ 0.5": [
        accuracy_score(y_test, (lr_probs >= 0.5).astype(int)),
        accuracy_score(y_test, (lgb_probs >= 0.5).astype(int)),
        accuracy_score(y_test, (cal_probs >= 0.5).astype(int))
    ]
}).set_index("Model")

comparison_df.round(4)


,Brier Score (lower is better),Log Loss (lower is better),ROC-AUC (higher is better),Accuracy @ 0.5
Model,,,,
Logistic Regression,0.1887,0.6167,0.8357,0.7442
Raw LightGBM,0.2139,0.8140,0.8224,0.7314
Calibrated LightGBM,0.1781,0.7800,0.8403,0.7528


## 7. Innings 1 Modeling (Score Projection & Defense Win Probability)

In the 1st innings, there is no target yet. We train two models:
1. **LightGBM Regressor:** Predicts expected final 1st innings total (`final_innings1_score`) based on current score, wickets in hand, overs completed, and venue.
2. **Isotonic Calibrated LightGBM Classifier:** Predicts the probability that the 1st batting team will successfully defend their total.

In [8]:
from sklearn.metrics import mean_absolute_error, r2_score

df_inn1 = pd.read_csv('../data/processed/innings1_ballbyball.csv', low_memory=False)

def extract_year(s):
    s_str = str(s).strip()
    if '/' in s_str: return int(s_str.split('/')[0])
    try: return int(s_str)
    except: return 2008

years_inn1 = df_inn1['season'].apply(extract_year)
df_train1 = df_inn1[years_inn1 < 2023].copy()
df_test1 = df_inn1[years_inn1 >= 2023].copy()

num_cols1 = ['current_score', 'legal_balls_bowled', 'balls_left', 'wickets_fallen', 'wickets_in_hand', 'crr']
cat_cols1 = ['batting_team', 'bowling_team', 'venue']
feat_cols1 = num_cols1 + cat_cols1

X_train1, y_train_win1 = df_train1[feat_cols1], df_train1['batting_team_won']
X_test1, y_test_win1 = df_test1[feat_cols1], df_test1['batting_team_won']
y_train_score1 = df_train1['final_innings1_score']
y_test_score1 = df_test1['final_innings1_score']

prep1 = ColumnTransformer([
    ('num', 'passthrough', num_cols1),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols1)
])

# 1. Innings 1 Score Projection Regressor
pipe_reg1 = Pipeline([
    ('prep', prep1),
    ('reg', lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1))
])
pipe_reg1.fit(X_train1, y_train_score1)
preds_score1 = pipe_reg1.predict(X_test1)

print('--- Innings 1 Score Regressor Performance ---')
print(f'Mean Absolute Error (MAE): {mean_absolute_error(y_test_score1, preds_score1):.2f} runs')
print(f'R² Score:                 {r2_score(y_test_score1, preds_score1):.4f}')

# 2. Innings 1 Defense Win Classifier
pipe_cls1 = Pipeline([
    ('prep', prep1),
    ('clf', lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1))
])
cal_cls1 = CalibratedClassifierCV(pipe_cls1, method='isotonic', cv=3)
cal_cls1.fit(X_train1, y_train_win1)
probs_win1 = cal_cls1.predict_proba(X_test1)[:, 1]

print('\n--- Innings 1 Defense Win Classifier Performance ---')
print(f'Brier Score:              {brier_score_loss(y_test_win1, probs_win1):.4f}')
print(f'ROC-AUC:                  {roc_auc_score(y_test_win1, probs_win1):.4f}')


--- Innings 1 Score Regressor Performance ---
Mean Absolute Error (MAE): 20.68 runs
R² Score:                 0.3918



--- Innings 1 Defense Win Classifier Performance ---
Brier Score:              0.2476
ROC-AUC:                  0.5957
